In [24]:
"""
NB02_Frames_and_Nominal_Orbit_Generator

Purpose
-------
1) Generate a nominal LLO reference orbit sampled uniformly in true anomaly nu.
   Supports arbitrary inclination; defaults to CONFIG polar orbit (i=90 deg).
2) Provide inertial <-> Moon-fixed rotation using constant lunar spin about +Z.
3) Provide RTN frame construction from (r, v).

Outputs (in-memory)
-------------------
- nu_grid, t_grid
- r_eci[ M x 3 ], v_eci[ M x 3 ]
- r_fixed[ M x 3 ]
- RTN basis (rhat, that, nhat) each [ M x 3 ]
- Helper rotation utilities: R1, R3, inertial_to_fixed, fixed_to_inertial

Dependencies
------------
Expects CONFIG dict from NB00 (via %run or shared kernel).
Falls back to NB00-consistent constants with a warning if CONFIG is absent.
"""

'\nNB02_Frames_and_Nominal_Orbit_Generator\n\nPurpose\n-------\n1) Generate a nominal LLO reference orbit sampled uniformly in true anomaly nu.\n   Supports arbitrary inclination; defaults to CONFIG polar orbit (i=90 deg).\n2) Provide inertial <-> Moon-fixed rotation using constant lunar spin about +Z.\n3) Provide RTN frame construction from (r, v).\n\nOutputs (in-memory)\n-------------------\n- nu_grid, t_grid\n- r_eci[ M x 3 ], v_eci[ M x 3 ]\n- r_fixed[ M x 3 ]\n- RTN basis (rhat, that, nhat) each [ M x 3 ]\n- Helper rotation utilities: R1, R3, inertial_to_fixed, fixed_to_inertial\n\nDependencies\n------------\nExpects CONFIG dict from NB00 (via %run or shared kernel).\nFalls back to NB00-consistent constants with a warning if CONFIG is absent.\n'

In [25]:
# ============================================================
# Cell 1 : Imports and Constants from NB00
# ============================================================
import numpy as np
import warnings

# Pull from NB00 CONFIG if available; otherwise use NB00-consistent fallbacks.
try:
    mu          = CONFIG["mu_km3_s2"]
    Rm          = CONFIG["R_moon_km"]
    omega_moon  = CONFIG["omega_moon_rad_s"]
    M_default   = CONFIG["M_default"]
    _default_orbit = CONFIG["default_orbit"]
    _config_source = "CONFIG (NB00)"
except NameError:
    warnings.warn(
        "CONFIG not found — using NB00-consistent fallback constants. "
        "Run NB00 first for full reproducibility.",
        stacklevel=2
    )
    # Fallback values: MUST match NB00 exactly.
    mu          = 4902.800076        # [km^3/s^2] GRAIL GL0900D
    Rm          = 1737.4             # [km]
    omega_moon  = 2.6616995e-6       # [rad/s]
    M_default   = 4000
    _default_orbit = {
        "a_km":      Rm + 50.0,
        "e":         0.0,
        "i_deg":     90.0,
        "RAAN_deg":  0.0,
        "omega_deg": 0.0,
        "nu0_deg":   0.0,
    }
    _config_source = "FALLBACK (NB00-consistent)"

print(f"Source         : {_config_source}")
print(f"mu [km^3/s^2]  = {mu}")
print(f"R_moon [km]    = {Rm}")
print(f"omega [rad/s]  = {omega_moon:.7e}")
print(f"M_default      = {M_default}")
print(f"Default orbit  = {_default_orbit}")

Source         : FALLBACK (NB00-consistent)
mu [km^3/s^2]  = 4902.800076
R_moon [km]    = 1737.4
omega [rad/s]  = 2.6616995e-06
M_default      = 4000
Default orbit  = {'a_km': 1787.4, 'e': 0.0, 'i_deg': 90.0, 'RAAN_deg': 0.0, 'omega_deg': 0.0, 'nu0_deg': 0.0}


In [26]:
# ============================================================
# Cell 2 : Linear Algebra Helpers
# ============================================================
def norm_rows(X: np.ndarray) -> np.ndarray:
    """Row-wise Euclidean norm for an (N,3) array."""
    return np.linalg.norm(X, axis=1)


def unit_rows(X: np.ndarray) -> np.ndarray:
    """Row-wise unit vectors for an (N,3) array."""
    n = norm_rows(X)
    if np.any(n == 0):
        raise ValueError("Zero-norm row encountered in unit_rows().")
    return X / n[:, None]

In [27]:
# ============================================================
# Cell 3 : Rotation Matrices and Frame Transforms
# ============================================================
def R1(theta: float) -> np.ndarray:
    """Rotation matrix about +X by angle theta (right-hand rule)."""
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[1.0,  0.0, 0.0],
                     [0.0,   c,  -s ],
                     [0.0,   s,   c ]], dtype=float)


def R3(theta: float) -> np.ndarray:
    """Rotation matrix about +Z by angle theta (right-hand rule)."""
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[ c,  -s, 0.0],
                     [ s,   c, 0.0],
                     [0.0, 0.0, 1.0]], dtype=float)


def inertial_to_fixed(r_eci: np.ndarray, t: np.ndarray, omega: float) -> np.ndarray:
    """
    Rotate inertial position(s) into Moon-fixed frame.

    Convention
    ----------
    The Moon-fixed frame rotates at +omega about inertial +Z.
    At time t the Moon-fixed x-axis is at angle +omega*t in inertial space.
    To express an inertial vector in the fixed frame we undo that rotation:

        r_fixed = R3(-omega * t) @ r_inertial

    Parameters
    ----------
    r_eci : (N, 3)  positions in the inertial frame [km]
    t     : (N,)    time values [s]
    omega : float   lunar spin rate [rad/s]

    Returns
    -------
    r_fixed : (N, 3) positions in the Moon-fixed frame [km]
    """
    r_fixed = np.empty_like(r_eci)
    for k in range(r_eci.shape[0]):
        r_fixed[k] = R3(-omega * t[k]) @ r_eci[k]
    return r_fixed


def fixed_to_inertial(vec_fixed: np.ndarray, t: np.ndarray, omega: float) -> np.ndarray:
    """
    Rotate Moon-fixed vector(s) back to the inertial frame.

        vec_inertial = R3(+omega * t) @ vec_fixed

    Parameters
    ----------
    vec_fixed : (N, 3)
    t         : (N,)
    omega     : float

    Returns
    -------
    vec_eci : (N, 3)
    """
    vec_eci = np.empty_like(vec_fixed)
    for k in range(vec_fixed.shape[0]):
        vec_eci[k] = R3(+omega * t[k]) @ vec_fixed[k]
    return vec_eci

In [28]:
# ============================================================
# Cell 4 : RTN Basis Construction
# ============================================================
def rtn_basis_from_state(r_eci: np.ndarray, v_eci: np.ndarray):
    """
    Build RTN basis vectors (rhat, that, nhat) for each state.

    rhat : radial unit vector           (along r)
    nhat : orbit-normal unit vector      (along h = r x v)
    that : transverse unit vector        (nhat x rhat), completing
           a right-handed triad (R, T, N)

    Inputs : r_eci, v_eci  (N, 3)
    Outputs: rhat, that, nhat  (N, 3)
    """
    rhat = unit_rows(r_eci)
    h    = np.cross(r_eci, v_eci)
    nhat = unit_rows(h)
    that = np.cross(nhat, rhat)
    that = unit_rows(that)          # re-normalise for numerical hygiene

    # Orthonormality diagnostics
    ortho_rt = np.max(np.abs(np.sum(rhat * that, axis=1)))
    ortho_rn = np.max(np.abs(np.sum(rhat * nhat, axis=1)))
    ortho_tn = np.max(np.abs(np.sum(that * nhat, axis=1)))
    print(f"Orthonormality  max|R·T|={ortho_rt:.2e}  max|R·N|={ortho_rn:.2e}  max|T·N|={ortho_tn:.2e}")

    return rhat, that, nhat

In [29]:
# ============================================================
# Cell 5 : Nominal Orbit Generator (arbitrary inclination)
# ============================================================
def generate_nominal_orbit(
    M: int,
    mu: float,
    a_km: float,
    e: float       = 0.0,
    i_deg: float   = 90.0,
    RAAN_deg: float  = 0.0,
    omega_deg: float = 0.0,
    nu0_deg: float   = 0.0,
):
    """
    Build a Keplerian reference orbit sampled uniformly in true anomaly.

    For the single-orbit encounter mapping (Section 3.1) the orbit is
    treated as a fixed nominal reference; mascon effects are perturbations
    about this reference.

    Parameters
    ----------
    M          : int     number of anomaly samples (one revolution)
    mu         : float   gravitational parameter [km^3/s^2]
    a_km       : float   semimajor axis [km]
    e          : float   eccentricity (0 = circular)
    i_deg      : float   inclination [degrees]
    RAAN_deg   : float   right ascension of ascending node [degrees]
    omega_deg  : float   argument of periapsis [degrees]
    nu0_deg    : float   starting true anomaly [degrees]

    Returns
    -------
    dict with keys:
        nu, t, r_eci, v_eci, a, e, i_rad,
        RAAN_rad, omega_rad, p, h, v_circ,
        n_mean, T
    """
    # Convert angles to radians
    i_rad    = np.radians(i_deg)
    RAAN_rad = np.radians(RAAN_deg)
    om_rad   = np.radians(omega_deg)
    nu0_rad  = np.radians(nu0_deg)

    # Orbital geometry
    p = a_km * (1.0 - e**2)          # semi-latus rectum
    h = np.sqrt(mu * p)              # specific angular momentum
    n_mean = np.sqrt(mu / a_km**3)   # mean motion
    T = 2.0 * np.pi / n_mean         # period

    # True anomaly grid: one full revolution starting from nu0
    nu = nu0_rad + np.linspace(0.0, 2.0 * np.pi, M, endpoint=False)

    # Radius at each nu (general conic)
    r_mag = p / (1.0 + e * np.cos(nu - nu0_rad))  # note: argument relative to periapsis
    # For the general case, nu measured from periapsis direction:
    # here nu already includes nu0 offset, but e=0 makes this moot for circular.
    # Re-do properly: nu in the integral is true anomaly from periapsis.
    # Let f = nu - nu0_rad  (if omega=0 and nu0=0, f is true anomaly from periapsis)
    # For generality, the "true anomaly from periapsis" is simply (nu - om_rad) mod 2pi
    # but since we sample a full revolution it doesn't matter for circular.
    # For elliptical support: f = nu (since omega_deg offsets periapsis direction)
    f = nu  # true anomaly from periapsis (omega already rotates periapsis in the frame)
    r_mag = p / (1.0 + e * np.cos(f))

    # --- Perifocal frame (PQW) ---
    # P points to periapsis, Q is 90° ahead in the orbital plane, W = P x Q
    r_pqw = np.column_stack([
        r_mag * np.cos(f),
        r_mag * np.sin(f),
        np.zeros(M)
    ])

    v_pqw = np.column_stack([
        -(mu / h) * np.sin(f),
         (mu / h) * (e + np.cos(f)),
        np.zeros(M)
    ])

    # --- Rotation PQW -> ECI:  R3(-RAAN) R1(-i) R3(-omega) ---
    # Standard: r_eci = R3(-RAAN) @ R1(-i) @ R3(-omega) @ r_pqw
    Q_rot = R3(-RAAN_rad) @ R1(-i_rad) @ R3(-om_rad)

    r_eci = (Q_rot @ r_pqw.T).T    # (3,3)@(3,M) -> (3,M) -> transpose to (M,3)
    v_eci = (Q_rot @ v_pqw.T).T

    # --- Time grid ---
    # dt/df = r^2 / h  (Kepler's second law in anomaly domain)
    # For circular (e=0): dt/df = a^2 / h = const, so t = (a^2/h) * f
    # For elliptical: integrate numerically.
    dt_df = r_mag**2 / h
    df = f[1] - f[0]  # uniform spacing in f
    t = np.concatenate([[0.0], np.cumsum(dt_df[:-1] * df)])

    return {
        "nu": nu, "f": f, "t": t,
        "r_eci": r_eci, "v_eci": v_eci,
        "a": a_km, "e": e,
        "i_rad": i_rad, "RAAN_rad": RAAN_rad, "omega_rad": om_rad,
        "p": p, "h": h,
        "n_mean": n_mean, "T": T,
        "M": M,
    }

In [30]:
# ============================================================
# Cell 6 : Generate Default Orbit from CONFIG
# ============================================================
M = M_default

orb = generate_nominal_orbit(
    M         = M,
    mu        = mu,
    a_km      = _default_orbit["a_km"],
    e         = _default_orbit["e"],
    i_deg     = _default_orbit["i_deg"],
    RAAN_deg  = _default_orbit["RAAN_deg"],
    omega_deg = _default_orbit["omega_deg"],
    nu0_deg   = _default_orbit["nu0_deg"],
)

# Unpack for convenience
nu_grid = orb["nu"]
t_grid  = orb["t"]
r_eci   = orb["r_eci"]
v_eci   = orb["v_eci"]
a_km    = orb["a"]
e_val   = orb["e"]
i_rad   = orb["i_rad"]
p_km    = orb["p"]
h_spec  = orb["h"]
n_mean  = orb["n_mean"]
T_sec   = orb["T"]

print("Nominal orbit (from CONFIG default_orbit):")
print(f"  a [km]       = {a_km}")
print(f"  e            = {e_val}")
print(f"  i [deg]      = {np.degrees(i_rad)}")
print(f"  p [km]       = {p_km}")
print(f"  h [km^2/s]   = {h_spec:.6f}")
print(f"  v_circ [km/s]= {np.sqrt(mu / a_km):.10f}")
print(f"  n [rad/s]    = {n_mean:.12e}")
print(f"  T [s]        = {T_sec:.4f}  ({T_sec/3600:.4f} hrs)")
print(f"  M samples    = {M}")
print(f"  dnu [rad]    = {nu_grid[1]-nu_grid[0]:.10e}")
print(f"  dt/step [s]  ~ {t_grid[1]-t_grid[0]:.6f}")

Nominal orbit (from CONFIG default_orbit):
  a [km]       = 1787.4
  e            = 0.0
  i [deg]      = 90.0
  p [km]       = 1787.4
  h [km^2/s]   = 2960.281212
  v_circ [km/s]= 1.6561940317
  n [rad/s]    = 9.265939530698e-04
  T [s]        = 6780.9479  (1.8836 hrs)
  M samples    = 4000
  dnu [rad]    = 1.5707963268e-03
  dt/step [s]  ~ 1.695237


In [31]:
# ============================================================
# Cell 7 : RTN Basis and State-Vector Sanity Checks
# ============================================================
rhat, that, nhat = rtn_basis_from_state(r_eci, v_eci)

r_norm = norm_rows(r_eci)
v_norm = norm_rows(v_eci)

print(f"r_norm  min/max: {r_norm.min():.10f} / {r_norm.max():.10f}  (expect {a_km})")
print(f"v_norm  min/max: {v_norm.min():.10f} / {v_norm.max():.10f}")

# Angular momentum direction
h_vec = np.cross(r_eci, v_eci)
h_hat = unit_rows(h_vec)
print(f"h_hat mean : {h_hat.mean(axis=0)}")
print(f"h_hat std  : {h_hat.std(axis=0)}")

# For a polar orbit (i=90 deg, RAAN=0), h should point along -Y in ECI
# (h = r x v, with ascending node at +X and orbit going over +Z pole)
if abs(np.degrees(i_rad) - 90.0) < 1.0:
    print("(Polar orbit: h_hat should be roughly [0, +1, 0] for RAAN=0)")

Orthonormality  max|R·T|=5.55e-17  max|R·N|=2.47e-32  max|T·N|=1.23e-32
r_norm  min/max: 1787.4000000000 / 1787.4000000000  (expect 1787.4)
v_norm  min/max: 1.6561940317 / 1.6561940317
h_hat mean : [3.94601173e-35 1.00000000e+00 6.12323400e-17]
h_hat std  : [1.92769692e-33 0.00000000e+00 6.68284783e-30]
(Polar orbit: h_hat should be roughly [0, +1, 0] for RAAN=0)


In [32]:
# ============================================================
# Cell 8 : Frame Rotation Test (inertial <-> fixed round-trip)
# ============================================================
r_fixed = inertial_to_fixed(r_eci, t_grid, omega=omega_moon)

# Magnitudes must be preserved
r_fixed_norm = norm_rows(r_fixed)
print(f"Fixed-frame r_norm  min/max: {r_fixed_norm.min():.10f} / {r_fixed_norm.max():.10f}")

# Round-trip test: fixed -> inertial should recover r_eci
r_roundtrip = fixed_to_inertial(r_fixed, t_grid, omega=omega_moon)
roundtrip_err = np.max(np.abs(r_roundtrip - r_eci))
print(f"Round-trip max error [km]: {roundtrip_err:.2e}  (expect ~machine eps * r)")

# Show first few samples
print("\nr_eci[0:3]:")
print(r_eci[:3])
print("r_fixed[0:3]:")
print(r_fixed[:3])

Fixed-frame r_norm  min/max: 1787.4000000000 / 1787.4000000000
Round-trip max error [km]: 2.27e-13  (expect ~machine eps * r)

r_eci[0:3]:
[[ 1.78740000e+03  0.00000000e+00  0.00000000e+00]
 [ 1.78739779e+03  1.71918379e-16 -2.80764020e+00]
 [ 1.78739118e+03  3.43836334e-16 -5.61527347e+00]]
r_fixed[0:3]:
[[ 1.78740000e+03  0.00000000e+00  0.00000000e+00]
 [ 1.78739779e+03 -8.06511671e-03 -2.80764020e+00]
 [ 1.78739118e+03 -1.61301737e-02 -5.61527347e+00]]


In [33]:
# ============================================================
# Cell 9 : Save Nominal Orbit Data
# ============================================================
alt_km = a_km - Rm
run_tag = f"llo{int(alt_km)}km_M{M}"
npz_name = f"nominal_orbit_{run_tag}.npz"

np.savez(
    npz_name,
    # Grid
    nu=nu_grid,
    t=t_grid,
    # States
    r_eci=r_eci,
    v_eci=v_eci,
    r_fixed=r_fixed,
    # RTN basis
    rhat=rhat,
    that=that,
    nhat=nhat,
    # Orbital elements
    mu=mu,
    Rm=Rm,
    omega_moon=omega_moon,
    a=a_km,
    e=e_val,
    i_rad=i_rad,
    RAAN_rad=orb["RAAN_rad"],
    omega_rad=orb["omega_rad"],
    p=p_km,
    h=h_spec,
    n_mean=n_mean,
    period_s=T_sec,
)

print(f"Saved: {npz_name}")
print(f"\n--- NB02 complete ---")

Saved: nominal_orbit_llo50km_M4000.npz

--- NB02 complete ---
